# Quick overview of yabadaba

Yabadaba makes it easy to track Python data and metadata so that it can be stored in and retrieved from databases. This helps make data more accessible as it allows for interacting with public databases and automatic interpretation of the data into meaningful Python objects.

The steps for incorporating new data schemas into yabadaba are

1. Decide what (meta)data fields you want to capture.
2. Define a Record class containing Value definitions for all of those important data fields.
3. Tell yabadaba about the new Record class.

Once a record is integrated with yabadaba, you can 

4. Use yabadaba to build, interpret and transform records, and
5. Interact with databases containing such records.

In [1]:
import yabadaba

## 1. Decide what (meta)data fields you want to capture

Before creating a Record, it can be useful to think about what data and metadata you wish to capture. Some general guidance

- The data fields in a Record class should focus on simple values that users may wish to query, parse, or sort by. It is best to store any associated large "raw" data elsewhere such that it can be retrieved *after* records of interest are selected. This speeds up query operations and reduces memory usage.
- As for metadata, the recommendation is for inclusion over exclusion. Include as many simple metadata fields as you can think of as even if they do not seem important to you now they may be important in the future or to someone else.
- Reuse schemas or schema parts if possible rather than reinventing the wheel. Check if there are any existing schemas for related data to use or gain inspiration from. If you plan on defining multiple Record classes, are there common data subsets that they share?
- Tree-like data formats like JSON and XML allow for the data to be "flat" or "tiered". Flat schemas are simpler, but tiered schemas group related values together for context. The tiered schemas are also are more suited for reusing common subsets across different schemas and for representing repeated subsets within a single schema.
- What type of data are the data fields (numbers, strings, etc?) Do numerical values need units? Should the values be interpreted independently or should a subset of values be interpreted together (like input parameters for creating a Python object)?

## 2. Define a Record subclass

The primary steps for defining a Record subclass are

1. Create a Python class definition that inherits from yabadaba.record.Record.
2. Define class properties "style" and "modelroot". "style" is the style name that yabadaba will refer to the record class by. "modelroot" is the name of the single root element to use when generating the JSON/XML data.
3. Define an _init_values(self) method. Inside the method, make multiple calls to self._add_value() to define all data values for inclusion in the schema.  Note that the order that you define the values is the order that the values will appear in the JSON/XML content. The two primary arguments when defining a value are the value's style and its name.

Below, we'll define a simple (incomplete) example record for XSPF music playlist files (https://www.xspf.org/). Two Record classes are defined: XSPF and XSPFTrack. Each XSPF can contain multiple XSPFTracks, which is indicated by creating a Value with style 'recordlist' that points to the XSPFTrack class.

In [2]:
from yabadaba.record import Record

class XSPFTrack(Record):
    """yabadaba translator for XSPF playlist track elements"""

    ########################## Basic metadata fields ##########################

    @property
    def style(self):
        """str: The record style"""
        return 'xspf_track'

    @property
    def modelroot(self):
        """str: The root element of the content"""
        return 'track'

    ############################# Define Values  ##############################
    
    def _init_values(self):
        """
        Method that defines the value objects for the Record.  This should
        call the super of this method, then use self._add_value to create new Value objects.
        Note that the order values are defined matters
        when build_model is called!!!
        """
        self._add_value('str', 'location', description='file path or URL where the track can be found')
        self._add_value('str', 'title', description='name of the track')
        self._add_value('str', 'creator', description="name of the entity that created the track (author, artist, etc.)")
        self._add_value('str', 'album', description='name of the collection from which the track originated')

        # Modify query names to avoid naming conflicts with XSPF values
        for key in ['title', 'creator']:
            valobj = self.get_value(key)
            valobj.queries[f'track_{key}'] = valobj.queries.pop(key)
        

class XSPF(Record):
    """yabadaba translator of XSPF playlist files"""

    ########################## Basic metadata fields ##########################

    @property
    def style(self):
        """str: The record style"""
        return 'xspf'

    @property
    def modelroot(self):
        """str: The root element of the content"""
        return 'playlist'

    ############################# Define Values  ##############################

    def _init_values(self):
        """
        Method that defines the value objects for the Record.  This should
        call the super of this method, then use self._add_value to create new Value objects.
        Note that the order values are defined matters
        when build_model is called!!!
        """
        self._add_value('longstr', 'title', description='name of the playlist')
        self._add_value('str', 'creator', description='name of the person who created the playlist')
        self._add_value('recordlist', 'track', recordclass=XSPFTrack, description='the list of tracks')


## 3. Tell yabadaba about the new Record class.

By subclassing Record, objects of XSPF allow for interacting and transforming the data but the database APIs are not aware of it yet. To allow this, the Record needs to be added to yabadaba's record manager.

The two ways to add a record to recordmanager:
- The simple way is to add the record to the recordmanager.loaded_styles dict.
- The modular way is to use recordmanager.import_style(). This adds the record style if it can be successfully imported, and captures the error message if not. Using import_style() lets records have more package requirements than yabadaba without breaking yabadaba entirely.

In [3]:
# Direct assignment to the recordmanager.loaded_style dict 
# key is the record's style name, value is the record class
yabadaba.recordmanager.loaded_styles['xspf'] = XSPF

## 4. Use yabadaba to build, interpret and transform records

### 4.1. Creating a new record

New record objects can be created either directly like any other Python object or by calling __yabadaba.load_record()__ and specifying the record's style.

In [4]:
# Demonstrate that calling load_record('xspf') creates a new XSPF object
new_xspf = yabadaba.load_record('xspf')
new_xspf

### 4.2. Getting and setting values

All Values that were defined in the class are automatically recognized by name both as keyword parameters during initialization and as class attributes. This allows for values to be set both during initialization or anytime afterwards.

**NOTE**: All records also have a "name" attribute that ususally needs to be set that corresponds to what the record would be called when saved as a file or stored in a database.

In [5]:
# Create a new XSPF called 'demo', and set title
new_xspf = yabadaba.load_record('xspf', name='demo', title='demo playlist')

# Directly set creator value after init
new_xspf.creator = 'me'

# Check title and creator values
print(f'Playlist "{new_xspf.title}" created by {new_xspf.creator}')

Playlist "demo playlist" created by me


### 4.3. Data transformations

There are three distinct representations for the Record content 
- The "Python" representation in which Values are class attributes, as shown above,
- The "data model" representation of embedded dicts consistent with the tree-like format of JSON/XML, and
- The "metadata" representation of a flat dict containing simple values from the record.

The record data can be transformed between these different representations using the following methods
- build_model() creates the data model representation based on the current values of Value attributes.
- reload_model() updates the values of Value attributes based on the current data model values.
- metadata() generates the flat metadata representation based on the current values of Value attributes.


In [6]:
# Call build_model() to construct the data model representation
# The model representation is both returned and set as the model attribute!
model = new_xspf.build_model()

# Show as XML
print(model.xml(indent=2))
print()

# Show as JSON
print(new_xspf.model.json(indent=2))
print()

# Generate and show the metadata representation
print(new_xspf.metadata())

<?xml version="1.0" encoding="utf-8"?>
<playlist>
  <title>demo playlist</title>
  <creator>me</creator>
</playlist>

{
  "playlist": {
    "title": "demo playlist",
    "creator": "me"
  }
}

{'name': 'demo', 'title': 'demo playlist', 'creator': 'me', 'track': []}


### 4.4. Special 'recordlist' values

The 'recordlist' value style is special as it represents a list of Record objects of a different class. A number of features are automatically added in support of such values.

- The subset values can be set during initialization by passing in a list of dicts
- A method called "add_\<valuename\>()" will be created that conveniently allows new subsets to be added one at a time
- A method called "\<valuename\>_df()" will be created that generates a pandas DataFrame based on the metadata representations of the subsets.

In [7]:
# Create a new XSPF with tracks
new_xspf = yabadaba.load_record('xspf', name='Halloween', title='Halloween', creator='Harry Goule',
                               track = [
                                   {'location':'file://music/song_1.mp3', 'title': "I'm a Changin'", 'creator':'Billy and the Werewolves', 'album': 'Moonglow'},
                                   {'location':'file://music/song_2.mp3', 'title': "Be Positive", 'creator':'Dr. Acula', 'album': 'Bats!'}
                               ])

# Add another track with add_track()
new_xspf.add_track(location='file://music/song_3.mp3', title='The Curse of Love', creator='Tres Wytches', album='Spelly Cat')

# The attribute "track" is the list of XSPFTrack objects
print(new_xspf.track)

# Calling track_df() will display a DataFrame of the track metadata
new_xspf.track_df()

[<__main__.XSPFTrack object at 0x000001B313095F30>, <__main__.XSPFTrack object at 0x000001B313095E00>, <__main__.XSPFTrack object at 0x000001B31318C4D0>]


,location,title,creator,album
0,file://music/song_1.mp3,I'm a Changin',Billy and the Werewolves,Moonglow
1,file://music/song_2.mp3,Be Positive,Dr. Acula,Bats!
2,file://music/song_3.mp3,The Curse of Love,Tres Wytches,Spelly Cat


Show the full record as converted to XML

In [8]:
print(new_xspf.build_model().xml(indent=2))

<?xml version="1.0" encoding="utf-8"?>
<playlist>
  <title>Halloween</title>
  <creator>Harry Goule</creator>
  <track>
    <location>file://music/song_1.mp3</location>
    <title>I'm a Changin'</title>
    <creator>Billy and the Werewolves</creator>
    <album>Moonglow</album>
  </track>
  <track>
    <location>file://music/song_2.mp3</location>
    <title>Be Positive</title>
    <creator>Dr. Acula</creator>
    <album>Bats!</album>
  </track>
  <track>
    <location>file://music/song_3.mp3</location>
    <title>The Curse of Love</title>
    <creator>Tres Wytches</creator>
    <album>Spelly Cat</album>
  </track>
</playlist>


## 5. Interact with databases containing such records

Now that yabadaba knows about the record class, we can create a database object where such records can be stored, and later queried and retrieved from.

Yabadaba currently supports three distinct database infrastructure styles: 
- 'local' for local folders containing XML or JSON files
- 'mongo' for MongoDB Databases
- 'cdcs' for CDCS (curator) Databases

You can initialize a database with yabadaba.load_database(), specify the style, and access parameters.  For 'local' styles, the access parameters are the directory path and optionally the file format.  For 'mongo', the access parameters correspond to initializing a [pymongo.MongoClient](https://pymongo.readthedocs.io/en/stable/index.html) object. For 'cdcs', the access parameters correspond to initializing a [cdcs.CDCS](https://github.com/usnistgov/pycdcs) object.

In [9]:
# Initialize a local database in the playlists directory
database = yabadaba.load_database(style='local', host='playlists', format='xml')

All database objects feature the same basic APIs for interacting with records in the database. For instance,

- add_record() will upload a new record to the database.
- update_record() will update the content of a record in the database.
- get_records() will query and return all matching records.
- count_records() will query and return a count of all matching records.
- delete_record() will delete a record from the database.

In [10]:
# Add the above record to the database
database.add_record(record=new_xspf)

In [11]:
# Get all XSPF records (only the one at the moment...)
# Setting return_df=True will also return a pandas DataFrame of the metadata
records, records_df = database.get_records('xspf', return_df=True)

# Show the first record
print(records[0].model.json(indent=2))

# Show the metadata DataFrame
records_df

{
  "playlist": {
    "title": "Halloween",
    "creator": "Harry Goule",
    "track": [
      {
        "location": "file://music/song_1.mp3",
        "title": "I'm a Changin'",
        "creator": "Billy and the Werewolves",
        "album": "Moonglow"
      },
      {
        "location": "file://music/song_2.mp3",
        "title": "Be Positive",
        "creator": "Dr. Acula",
        "album": "Bats!"
      },
      {
        "location": "file://music/song_3.mp3",
        "title": "The Curse of Love",
        "creator": "Tres Wytches",
        "album": "Spelly Cat"
      }
    ]
  }
}


,name,title,creator,track
0,Halloween,Halloween,Harry Goule,"[{'location': 'file://music/song_1.mp3', 'titl..."


When calling get_records() or count_records(), nearly every Value of the specified record style is a queryable keyword!  You can check queryable terms using Record.querydoc() or Database.record_querydoc().

In [16]:
print(database.record_querydoc('xspf'))

# xspf Query Parameters

- __title__ (*str or list, optional*): Return only the records where name of the playlist contains the given values
- __creator__ (*str or list, optional*): Return only the records where name of the person who created the playlist matches a given value
- __location__ (*str or list, optional*): Return only the records where file path or URL where the track can be found matches a given value
- __track_title__ (*str or list, optional*): Return only the records where name of the track matches a given value
- __track_creator__ (*str or list, optional*): Return only the records where name of the entity that created the track (author, artist, etc.) matches a given value
- __album__ (*str or list, optional*): Return only the records where name of the collection from which the track originated matches a given value



In [18]:
print(database.count_records('xspf'), 'total xspf records')
print(database.count_records('xspf', creator='me'), 'xspf records with creator "me"')
print(database.count_records('xspf', creator="Harry Goule"), 'xspf records with creator "Harry Goule"')

1 total xspf records
0 xspf records with creator "me"
1 xspf records with creator "Harry Goule"


In [19]:
# Delete record and show database to be empty
database.delete_record(record=new_xspf)
print(database.count_records('xspf'), 'total xspf records')

0 total xspf records
